# 05a — DeepLabV3+ (ResNet-50)

**Paper**: Mohammad et al. — *Deep Learning based Semantic Segmentation for Mars Rover Terrain Classification*  
DOI: `10.1109/iSpaRo60631.2024.10687827` — iSpaRo 2024  

**Arquitectura**: DeepLabV3+ con backbone ResNet-50 preentrenado en ImageNet, cabeza ASPP y cabeza auxiliar (weight=0.4).  
**Entrenamiento**: Google Colab Pro (A100 / T4 recomendado).

---

## Instrucciones de uso

### Prueba rapida en computador local (sin GPU potente)

Antes de lanzar el entrenamiento completo en Colab, verifica que el pipeline funciona en tu maquina local con `fast_subset=True`:

```python
# En la celda de configuracion, cambia:
FAST_SUBSET = True   # activa el subset pequeno (~200 imgs train, ~50 val)
```

Esto corre ~2 min por seed en CPU/RTX 4050 y verifica que no hay errores en el pipeline completo (carga de datos -> forward -> loss -> metricas -> checkpoint). Solo guarda el modelo del primer seed para no llenar disco.

**No uses** los resultados de `fast_subset=True` para el benchmark — son solo para depuracion.

### Entrenamiento completo en Google Colab Pro

1. Subir el proyecto a Google Drive en `/content/drive/MyDrive/ai4mars_DL-v3/`
2. Asegurarse de que `processed/split_indices_msl6k.pkl` este en el repo (no se regenera)
3. Montar Drive y ejecutar todas las celdas en orden
4. Los checkpoints se guardan en Drive automaticamente

## 0. Setup — Detectar entorno (Local vs Colab)

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/ai4mars_DL-v3')
    sys.path.append(str(PROJECT_ROOT / 'notebooks'))
    print('Colab — Drive montado')
else:
    # Local: este notebook esta en notebooks/, ROOT es el padre
    PROJECT_ROOT = Path('__file__').resolve().parent.parent
    # Fallback si __file__ no funciona en Jupyter
    if not (PROJECT_ROOT / 'processed').exists():
        PROJECT_ROOT = Path.cwd().parent
    print(f'Local — ROOT: {PROJECT_ROOT}')

print(f'ROOT existe: {PROJECT_ROOT.exists()}')
print(f'processed/ existe: {(PROJECT_ROOT / "processed").exists()}')
print(f'data/ existe: {(PROJECT_ROOT / "data").exists()}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from mars_utils import (
    set_seed, load_norm_stats, load_split,
    build_dataloaders, MetricsAccumulator,
    train_one_epoch, evaluate, train_model, run_multi_seed,
    append_benchmark_results, plot_best_seed_curves,
    print_summary_table, visualize_predictions, count_parameters,
    NUM_CLASSES, IGNORE_INDEX, SEEDS,
    CHECKPOINTS_DIR, BENCHMARK_CSV
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Configuracion

In [ ]:
# ======================================================
#  CONFIGURACION — ajustar segun entorno
# ======================================================

MODEL_NAME   = 'DeepLabV3+'
FAST_SUBSET  = True   # True para prueba rapida local; False para produccion en Colab

# Hiperparametros (segun KB seccion 9.1)
LR           = 0.001
MOMENTUM     = 0.9
WEIGHT_DECAY = 1e-4
AUX_WEIGHT   = 0.4
MAX_EPOCHS   = 80
PATIENCE     = 10     # KB v6: patience subido de 7 a 10
BATCH_SIZE   = 16

# Pesos de clase: soil=1.0, bedrock=0.8, sand=2.0, big_rock=8.0
CLASS_WEIGHTS = torch.tensor([1.0, 0.8, 2.0, 8.0], dtype=torch.float32).to(DEVICE)

print(f'Modo: {"FAST SUBSET (prueba)" if FAST_SUBSET else "PRODUCCION COMPLETA"}')
print(f'Epochs max: {MAX_EPOCHS} | Early stopping patience: {PATIENCE}')
print(f'Seeds: {SEEDS}')

## 2. Datos

In [ ]:
df_train, df_val, df_gold = load_split()
mean, std = load_norm_stats()

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Gold test: {len(df_gold)}')
print(f'Normalizacion — mean: {mean} | std: {std}')

In [ ]:
# Verificacion anti data leakage
# Bug 6: derivar stem de image_path — df_train['stem'] no existe
train_ids = set(Path(p).stem for p in df_train['image_path'])
gold_ids  = set(Path(p).stem for p in df_gold['image_path'])
assert len(train_ids & gold_ids) == 0, 'DATA LEAKAGE: overlap entre train y gold test'
print(f'Sin overlap entre train y gold test')
print(f'Train: {len(df_train)} | Val: {len(df_val)} | Gold: {len(df_gold)}')

## 3. Arquitectura — DeepLabV3+ con ResNet-50

In [ ]:
class DeepLabV3PlusMars(nn.Module):
    """
    DeepLabV3+ con backbone ResNet-50 preentrenado en ImageNet.
    - Salida principal: cabeza ASPP -> 4 clases
    - Salida auxiliar:  cabeza aux sobre layer3 -> 4 clases (weight=0.4 en loss)
    - Entrada esperada: [B, 3, 256, 256]
    - Salida en inferencia: [B, 4, 256, 256]

    Referencia: Mohammad et al., iSpaRo 2024.
    """
    def __init__(self, num_classes: int = 4, aux_loss: bool = True):
        super().__init__()
        self.aux_loss = aux_loss

        # Carga DeepLabV3 con ResNet-50 preentrenado y ajusta la cabeza final
        weights = DeepLabV3_ResNet50_Weights.DEFAULT
        base = deeplabv3_resnet50(weights=weights, aux_loss=aux_loss)

        # Reemplazar cabeza principal (256 canales ASPP -> num_classes)
        base.classifier[-1] = nn.Conv2d(256, num_classes, kernel_size=1)

        # Reemplazar cabeza auxiliar (256 -> num_classes)
        if aux_loss and base.aux_classifier is not None:
            base.aux_classifier[-1] = nn.Conv2d(256, num_classes, kernel_size=1)

        self.model = base

    def forward(self, x):
        h, w = x.shape[-2:]
        out = self.model(x)

        # Upsample a resolucion original si es necesario
        logits = F.interpolate(out['out'], size=(h, w),
                               mode='bilinear', align_corners=False)

        if self.training and self.aux_loss and 'aux' in out:
            aux_logits = F.interpolate(out['aux'], size=(h, w),
                                       mode='bilinear', align_corners=False)
            return logits, aux_logits
        return logits


def build_model():
    return DeepLabV3PlusMars(num_classes=NUM_CLASSES, aux_loss=True).to(DEVICE)


# Verificacion rapida de shapes
_m = build_model()
_m.eval()
with torch.no_grad():
    _x = torch.randn(2, 3, 256, 256).to(DEVICE)
    _out = _m(_x)
    print(f'Forward OK — salida eval: {_out.shape}')   # (2, 4, 256, 256)

_m.train()
with torch.no_grad():
    _out_train = _m(_x)
    print(f'Forward OK — salida train: logits={_out_train[0].shape}, aux={_out_train[1].shape}')

n_params = count_parameters(_m)
print(f'Parametros entrenables: {n_params:.2f}M  (referencia KB: ~42.00M)')
del _m, _x, _out, _out_train

## 4. Loss, Optimizer y Scheduler

In [ ]:
class DeepLabLoss(nn.Module):
    """
    CrossEntropyLoss con class weights e ignore_index.
    En entrenamiento combina salida principal + cabeza auxiliar (weight=0.4).
    En evaluacion solo usa la salida principal.
    """
    def __init__(self, weight: torch.Tensor, aux_weight: float = 0.4,
                 ignore_index: int = 255):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight, ignore_index=ignore_index)
        self.aux_weight = aux_weight

    def forward(self, output, target):
        if isinstance(output, (tuple, list)):
            logits, aux_logits = output
            loss_main = self.ce(logits, target)
            loss_aux  = self.ce(aux_logits, target)
            return loss_main + self.aux_weight * loss_aux
        return self.ce(output, target)


def criterion_fn():
    return DeepLabLoss(
        weight=CLASS_WEIGHTS,
        aux_weight=AUX_WEIGHT,
        ignore_index=IGNORE_INDEX
    )


def optimizer_fn(params):
    return torch.optim.SGD(
        params, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY
    )


def scheduler_fn(optimizer):
    # PolynomialLR: lr decae con potencia 0.9 a lo largo de MAX_EPOCHS iteraciones
    return torch.optim.lr_scheduler.PolynomialLR(
        optimizer, total_iters=MAX_EPOCHS, power=0.9
    )


print('Loss, optimizer y scheduler definidos')
print(f'  Loss: CrossEntropyLoss (weights=[1.0, 0.8, 2.0, 8.0], aux_weight={AUX_WEIGHT})')
print(f'  Optimizer: SGD (lr={LR}, momentum={MOMENTUM}, wd={WEIGHT_DECAY})')
print(f'  Scheduler: PolynomialLR (power=0.9, {MAX_EPOCHS} iters)')

## 5. Entrenamiento Multi-Seed

**Nota Colab**: Los checkpoints se guardan en `ROOT/checkpoints/` que apunta a Google Drive.  
**Nota local** con `FAST_SUBSET=True`: ~200 imgs train, ~50 val, ~2 min/seed — solo para verificar el pipeline.

In [ ]:
# Bug 7: run_multi_seed NO acepta mean, std ni max_epochs.
# El parametro correcto es num_epochs. mean/std los carga internamente.
summary = run_multi_seed(
    model_fn       = build_model,
    df_train       = df_train,
    df_val         = df_val,
    df_gold        = df_gold,
    criterion_fn   = criterion_fn,
    optimizer_fn   = optimizer_fn,
    scheduler_fn   = scheduler_fn,
    model_name     = MODEL_NAME,
    device         = DEVICE,
    num_epochs     = MAX_EPOCHS,   # 'num_epochs', no 'max_epochs'
    patience       = PATIENCE,
    batch_size     = BATCH_SIZE,
    fast_subset    = FAST_SUBSET,
    n_train_fast   = 200,
    n_val_fast     = 50,
)

## 6. Resultados Agregados

In [ ]:
print_summary_table(summary)

In [ ]:
plot_best_seed_curves(summary)

In [ ]:
# Bug 9: limpiar fila anterior del mismo modelo antes de guardar,
# para evitar duplicados si se reejecutar el notebook.
if BENCHMARK_CSV.exists():
    df_csv = pd.read_csv(BENCHMARK_CSV)
    df_csv = df_csv[df_csv['model'] != MODEL_NAME]
    df_csv.to_csv(BENCHMARK_CSV, index=False)

params_M = count_parameters(build_model())
append_benchmark_results(summary, params_M=params_M)
print(f'Resultados guardados en results/benchmark_results.csv')

## 7. Visualizacion Cualitativa

5 ejemplos del gold set: imagen original | ground truth | prediccion del mejor seed.

In [ ]:
# Bug 8: summary no tiene clave 'best_seed' ni 'checkpoints'.
# Extraer best_seed manualmente desde summary['per_seed'].
best_seed = max(summary['per_seed'], key=lambda r: r['mIoU'])['seed']
ckpt_path = CHECKPOINTS_DIR / f'{MODEL_NAME}_seed{best_seed}_best.pth'

best_model = build_model()
ckpt = torch.load(ckpt_path, map_location=DEVICE)
best_model.load_state_dict(ckpt['model_state'])
best_model.eval()

best_miou = max(summary['per_seed'], key=lambda r: r['mIoU'])['mIoU']
print(f'Mejor seed: {best_seed} | mIoU gold: {best_miou:.4f}')

visualize_predictions(
    model  = best_model,
    df     = df_gold,
    device = DEVICE,
    mean   = mean,
    std    = std,
    n      = 5
)

## 8. Resumen Final

| Campo | Valor |
|-------|-------|
| Modelo | DeepLabV3+ (ResNet-50) |
| Paper | Mohammad et al., iSpaRo 2024 |
| Backbone | ResNet-50 preentrenado ImageNet |
| Loss | CrossEntropyLoss + aux head (w=0.4) |
| Optimizer | SGD (lr=0.001, momentum=0.9) |
| Scheduler | PolynomialLR (power=0.9) |
| Referencia historica (2.1k imgs) | mIoU = 0.6806 +/- 0.0107 |

---
*Los resultados definitivos del gold set se exportaron a `results/benchmark_results.csv`.*